# EDA on 3-month data

In [0]:
from pyspark.sql.functions import col
import pyspark.sql.functions as F
import matplotlib.pyplot as plt
import seaborn as sns
from pyspark.sql.functions import count, rand
from pyspark.sql.types import *
from pyspark.sql.window import Window
import pandas as pd
import numpy as np


In [0]:
!pip install pydeck

In [0]:
import pydeck as pdk

In [0]:
data_BASE_DIR = "dbfs:/mnt/mids-w261/datasets_final_project_2022"
display(dbutils.fs.ls(f"{data_BASE_DIR}"))

In [0]:
# Airline Data    
df_flights = spark.read.parquet(f"dbfs:/mnt/mids-w261/datasets_final_project_2022/parquet_airlines_data_3m/")
df_flights = df_flights.cache()
_ = df_flights.count()  # materialize cache

## Data size and schema

In [0]:
n_rows = df_flights.count()
n_cols = len(df_flights.columns)

print(f"Rows: {n_rows:,}")
print(f"Columns: {n_cols:,}")

df_flights.printSchema()
display(df_flights.limit(10))

In [0]:
temporal_cols = [
    "YEAR","QUARTER","MONTH","DAY_OF_MONTH","DAY_OF_WEEK","FL_DATE"
]

airport_cols = [
    "ORIGIN_AIRPORT_ID", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID","ORIGIN","ORIGIN_CITY_NAME", "ORIGIN_STATE_ABR", "ORIGIN_STATE_FIPS","ORIGIN_STATE_NM","ORIGIN_WAC",
    "DEST_AIRPORT_ID","DEST_AIRPORT_SEQ_ID","DEST_CITY_MARKET_ID","DEST","DEST_CITY_NAME","DEST_STATE_ABR","DEST_STATE_FIPS","DEST_STATE_NM","DEST_WAC"
]

time_cols = [
    "CRS_DEP_TIME","DEP_TIME","DEP_TIME_BLK",
    "CRS_ARR_TIME","ARR_TIME","ARR_TIME_BLK",
    "TAXI_OUT","TAXI_IN","CRS_ELAPSED_TIME",
    "ACTUAL_ELAPSED_TIME","AIR_TIME", "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME"
]

#takeoff_landing_cols = ["WHEELS_OFF", "WHEELS_ON"]

carrier_cols = [
    "OP_UNIQUE_CARRIER","OP_CARRIER_AIRLINE_ID","OP_CARRIER","TAIL_NUM", "OP_CARRIER_FL_NUM"
]
target_col = ["DEP_DEL15"]

delay_cols = [
    "DEP_DELAY","DEP_DELAY_NEW","DEP_DELAY_GROUP",
    "ARR_DELAY","ARR_DELAY_NEW","ARR_DEL15", "ARR_DELAY_GROUP",
    "CARRIER_DELAY","WEATHER_DELAY","NAS_DELAY","SECURITY_DELAY","LATE_AIRCRAFT_DELAY"
]

distance_cols = [
    "DISTANCE","DISTANCE_GROUP"
]

operational_cols = [
    "CANCELLED","CANCELLATION_CODE",
    "DIVERTED","FLIGHTS"
]

diversion_cols = ["DIV_AIRPORT_LANDINGS", "DIV_REACHED_DEST", "DIV_ACTUAL_ELAPSED_TIME", "DIV_ARR_DELAY", "DIV_DISTANCE", "DIV1_AIRPORT", "DIV1_AIRPORT_ID", "DIV1_AIRPORT_SEQ_ID", "DIV1_WHEELS_ON", "DIV1_TOTAL_GTIME", "DIV1_LONGEST_GTIME", "DIV1_WHEELS_OFF", "DIV1_TAIL_NUM", "DIV2_AIRPORT", "DIV2_AIRPORT_ID", "DIV2_AIRPORT_SEQ_ID", "DIV2_WHEELS_ON", "DIV2_TOTAL_GTIME", "DIV2_LONGEST_GTIME", "DIV2_WHEELS_OFF", "DIV2_TAIL_NUM", "DIV3_AIRPORT", "DIV3_AIRPORT_ID", "DIV3_AIRPORT_SEQ_ID", "DIV3_WHEELS_ON", "DIV3_TOTAL_GTIME", "DIV3_LONGEST_GTIME", "DIV3_WHEELS_OFF", "DIV3_TAIL_NUM", "DIV4_AIRPORT", "DIV4_AIRPORT_ID", "DIV4_AIRPORT_SEQ_ID", "DIV4_WHEELS_ON", "DIV4_TOTAL_GTIME", "DIV4_LONGEST_GTIME", "DIV4_WHEELS_OFF", "DIV4_TAIL_NUM", "DIV5_AIRPORT", "DIV5_AIRPORT_ID", "DIV5_AIRPORT_SEQ_ID", "DIV5_WHEELS_ON", "DIV5_TOTAL_GTIME", "DIV5_LONGEST_GTIME", "DIV5_WHEELS_OFF", "DIV5_TAIL_NUM"]
    

print("Temporal:", len(temporal_cols),  temporal_cols)
print("Airport:", len(airport_cols),  airport_cols)
print("Carrier:", len(carrier_cols), carrier_cols)
print("Delay:", len(delay_cols), delay_cols)
print("Time:", len(time_cols), time_cols)
print("Distance:", len(distance_cols), distance_cols)
print("Operational:", len(operational_cols), operational_cols)
print("Diversion:", len(diversion_cols), diversion_cols)
#print("Takeoff-Landing:", len(takeoff_landing_cols), takeoff_landing_cols)

In [0]:
df_flights.select("FLIGHTS").distinct().show()

## Missing Value Analysis

Many delay columns are null when flights are not delayed or cancelled.

There are 47 columns with more than 99% of the rows being null values. These columns should be droped from the analysis as they will provide little to no-information to the model and, in case we decided to keep them, the imputation method might not work for the missing rows.


In [0]:
missing = (
    df_flights.select([
        F.count(F.when(F.col(c).isNull(), c)).alias(c)
        for c in df_flights.columns
    ])
)

missing_long = (
    missing.selectExpr(
        "stack(109," +
        ",".join([f"'{c}',{c}" for c in df_flights.columns]) +
        ") as (column, missing)"
    )
)

missing_long = missing_long.withColumn(
    "missing_pct",
    F.col("missing") / n_rows
)

display(
    missing_long.orderBy(F.desc("missing_pct"))
)

## Duplicates Analysis

Flights should be unique by date + carrier + flight number + origin + departure time.

In [0]:
dup_keys = ["FL_DATE",
            "OP_UNIQUE_CARRIER",
            "OP_CARRIER_FL_NUM",
            "ORIGIN",
            "DEST",
            "CRS_DEP_TIME"]

duplicates = (
    df_flights.groupBy(*dup_keys)
      .count()
      .filter("count > 1")
)

print("Duplicate flights:", duplicates.count())

In [0]:
distinct_rows = df_flights.distinct().count()
print("Distinct rows:", distinct_rows)

In [0]:
dup_flights = (
    df_flights
    .groupBy(dup_keys)
    .count()
    .filter(col("count") > 1)
    .orderBy(col("count").desc())
)

display(dup_flights)

In [0]:
sample_dup = dup_flights.limit(5)

sample_records = (
    df_flights.join(sample_dup, on=dup_keys, how="inner")
)

display(sample_records)

In [0]:
# validate whether there is a 1:1 mapping between OP_UNIQUE_CARRIER and OP_CARRIER_AIRLINE_ID

carrier_to_id = (
    df_flights
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(F.countDistinct("OP_CARRIER_AIRLINE_ID").alias("n_ids"))
)

#display(carrier_to_id.orderBy(F.desc("n_ids")))
carrier_violations = carrier_to_id.filter(F.col("n_ids") > 1)
display(carrier_violations)

id_to_carrier = (
    df_flights
    .groupBy("OP_CARRIER_AIRLINE_ID")
    .agg(F.countDistinct("OP_UNIQUE_CARRIER").alias("n_carriers"))
)

#display(id_to_carrier.orderBy(F.desc("n_carriers")))
id_violations = id_to_carrier.filter(F.col("n_carriers") > 1)
display(id_violations)

carrier_to_op = (
    df_flights
    .groupBy("OP_UNIQUE_CARRIER")
    .agg(F.countDistinct("OP_CARRIER").alias("n_ops"))
)

#display(carrier_to_op.orderBy(F.desc("n_ops")))
carrier_op_violations = carrier_to_op.filter(F.col("n_ops") > 1)
display(carrier_op_violations)


## Cardinality Analysis

In [0]:
cardinality = []

for col in df_flights.columns:
    distinct_count = df_flights.select(F.countDistinct(col)).collect()[0][0]
    cardinality.append((col, distinct_count))

cardinality_df = spark.createDataFrame(cardinality, ["column", "distinct_count"])

#cardinality_df.orderBy(F.desc("distinct_count")).show(truncate=False)

total_rows = df_flights.count()

# Percentage of uniqueness
cardinality_df = cardinality_df.withColumn(
    "pct_unique",
    F.col("distinct_count") / F.lit(total_rows)
)

In [0]:
display(
    cardinality_df.orderBy(F.desc("pct_unique"))
)

In [0]:
high_cardinality = cardinality_df.filter(F.col("pct_unique") > 0.001)
high_cardinality.show()

In [0]:
low_cardinality = cardinality_df.filter(F.col("distinct_count") <= 10)
low_cardinality.orderBy(F.desc("distinct_count")).show()

Columns that must be deleted for different reasons:
1. There is an equivalent column
2. There is a more granular column (for example: departure airport vs city name)
3. The column references to information that is collected after departure
4. Remove Operational and Diversion columns

In [0]:
operational_cols = [
    "CANCELLED","CANCELLATION_CODE",
    "DIVERTED","FLIGHTS"
]

diversion_cols = ["DIV_AIRPORT_LANDINGS", "DIV_REACHED_DEST", "DIV_ACTUAL_ELAPSED_TIME", "DIV_ARR_DELAY", "DIV_DISTANCE", "DIV1_AIRPORT", "DIV1_AIRPORT_ID", "DIV1_AIRPORT_SEQ_ID", "DIV1_WHEELS_ON", "DIV1_TOTAL_GTIME", "DIV1_LONGEST_GTIME", "DIV1_WHEELS_OFF", "DIV1_TAIL_NUM", "DIV2_AIRPORT", "DIV2_AIRPORT_ID", "DIV2_AIRPORT_SEQ_ID", "DIV2_WHEELS_ON", "DIV2_TOTAL_GTIME", "DIV2_LONGEST_GTIME", "DIV2_WHEELS_OFF", "DIV2_TAIL_NUM", "DIV3_AIRPORT", "DIV3_AIRPORT_ID", "DIV3_AIRPORT_SEQ_ID", "DIV3_WHEELS_ON", "DIV3_TOTAL_GTIME", "DIV3_LONGEST_GTIME", "DIV3_WHEELS_OFF", "DIV3_TAIL_NUM", "DIV4_AIRPORT", "DIV4_AIRPORT_ID", "DIV4_AIRPORT_SEQ_ID", "DIV4_WHEELS_ON", "DIV4_TOTAL_GTIME", "DIV4_LONGEST_GTIME", "DIV4_WHEELS_OFF", "DIV4_TAIL_NUM", "DIV5_AIRPORT", "DIV5_AIRPORT_ID", "DIV5_AIRPORT_SEQ_ID", "DIV5_WHEELS_ON", "DIV5_TOTAL_GTIME", "DIV5_LONGEST_GTIME", "DIV5_WHEELS_OFF", "DIV5_TAIL_NUM"]

delete_cols = ["ORIGIN_AIRPORT_ID", "ORIGIN_AIRPORT_SEQ_ID", "ORIGIN_CITY_MARKET_ID", "ORIGIN_CITY_NAME", "ORIGIN_STATE_NM", "ORIGIN_STATE_FIPS", "ORIGIN_WAC", "DEST_AIRPORT_ID", "DEST_AIRPORT_SEQ_ID", "DEST_CITY_MARKET_ID", "DEST_CITY_NAME", "DEST_STATE_NM", "DEST_STATE_FIPS", "DEST_WAC", "DEP_TIME","DEP_TIME_BLK", "ARR_TIME","ARR_TIME_BLK","TAXI_OUT","TAXI_IN","CRS_ELAPSED_TIME","ACTUAL_ELAPSED_TIME","AIR_TIME", "FIRST_DEP_TIME", "TOTAL_ADD_GTIME", "LONGEST_ADD_GTIME", "WHEELS_OFF", "WHEELS_ON"] + [ "OP_CARRIER", "OP_CARRIER_AIRLINE_ID", "OP_CARRIER_FL_NUM"] + ["DEP_DELAY_NEW","DEP_DELAY_GROUP","ARR_DELAY","ARR_DELAY_NEW","ARR_DEL15", "ARR_DELAY_GROUP","CARRIER_DELAY","WEATHER_DELAY","NAS_DELAY","SECURITY_DELAY","LATE_AIRCRAFT_DELAY"] + ["DISTANCE_GROUP"] + operational_cols + diversion_cols

len(delete_cols)

Prepare a usable dataset for analysis:
1. Deduplicating rows
2. Deleting unusable columns

## Clean Dataset

In [0]:

df_filtered = df_flights.filter(
    (F.col("CANCELLED") == 0) &
    (F.col("DIVERTED") == 0)
)
df_flights_dedup = df_filtered.dropDuplicates()
df_flights_usable = df_flights_dedup.drop(*delete_cols).cache()


In [0]:
display(
    df_flights_usable.limit(5)
)

## Delay Reasons

In [0]:
delay_cols = ["CARRIER_DELAY", "WEATHER_DELAY", "NAS_DELAY", "SECURITY_DELAY", "LATE_AIRCRAFT_DELAY"]

#NAS: National Air System Delay

df_delay = df_flights_dedup.filter(F.col("DEP_DEL15")==1).select(delay_cols)
df_delay = df_delay.toPandas()
df_delay[delay_cols] = df_delay[delay_cols].fillna(0)
df_delay['MAIN_DELAY_CAUSE'] = df_delay[delay_cols].idxmax(axis=1)

delay_counts = df_delay['MAIN_DELAY_CAUSE'].value_counts()
delay_pct = delay_counts / delay_counts.sum() * 100
delay_pct = delay_pct.reset_index()

In [0]:
dict_delay_cause = {"CARRIER_DELAY": "Carrier",
                    "LATE_AIRCRAFT_DELAY": "Late Aircraft",
                    "NAS_DELAY": "National Air System",
                    "WEATHER_DELAY": "Weather",
                    "SECURITY_DELAY": "Security"}

In [0]:
delay_pct

In [0]:
delay_pct["MAIN_DELAY_CAUSE"] = delay_pct["MAIN_DELAY_CAUSE"].map(dict_delay_cause)


In [0]:
delay_pct

In [0]:
values = delay_pct["count"]
labels = delay_pct["MAIN_DELAY_CAUSE"]

colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(values)))

plt.figure(figsize=(8, 8))

wedges, texts, autotexts = plt.pie(
    values,
    labels=labels,
    autopct='%1.1f%%',
    startangle=140,
    colors=colors,
    wedgeprops={'width': 0.4}
)

# Donut center
centre_circle = plt.Circle((0, 0), 0.70, fc='white')
plt.gca().add_artist(centre_circle)

plt.title('Flight Delays by Main Cause (%)', fontsize=14)

plt.tight_layout()
plt.show()

In [0]:
df_delay.head()

In [0]:
df_flights_usable.count()
#len(df_flights_usable.columns)

In [0]:
display(df_flights_usable.limit(10))

## Target Variable Exploration

The target variable DEP_DEL15, indicating whether a flight departed with a delay greater than 15 minutes, contained approximately 3% missing values. These null values correspond to flights that were cancelled or did not depart normally. Because cancelled flights represent a different operational outcome than departure delays, these observations should be excluded from the modeling dataset.

In [0]:
n_rows = df_flights_usable.count()
delay_dist = (
    df_flights_usable.groupBy("DEP_DEL15")
      .count()
      .withColumn("pct", F.col("count") / n_rows)
)

display(delay_dist)

delay_pd = delay_dist.toPandas()

plt.bar(delay_pd["DEP_DEL15"].astype(str), delay_pd["pct"])
plt.title("Flight Delay Distribution (>15 min)")
plt.ylabel("Percentage")
plt.xlabel("Delayed")
plt.show()

## Delay Distribution

In [0]:
delay_sample = (
    df_flights_usable.select("DEP_DELAY")
      .where("DEP_DELAY is not null")
      .sample(0.05)
      .toPandas()
)

plt.hist(delay_sample["DEP_DELAY"], bins=100)
plt.xlim(-50,300)
plt.title("Distribution of Departure Delay")
plt.xlabel("Minutes")
plt.ylabel("Frequency")
plt.show()

## Delay by Quarter

In [0]:

quarter_delay = (
    df_flights_usable.groupBy("QUARTER")
      .agg(
          F.count("*").alias("flights"),
          F.avg("DEP_DEL15").alias("delay_rate")
      )
      .orderBy("QUARTER")
)

# display(carrier_delay)

quarter_pd = quarter_delay.toPandas()

plt.figure(figsize=(10,5))
plt.bar(quarter_pd["QUARTER"].astype(str), quarter_pd["delay_rate"])
plt.title("Delay Rate by Quarter")
plt.ylabel("Delay Rate")
plt.xlabel("Quarter")
plt.show()

## Delay by Year

In [0]:

year_delay = (
    df_flights_usable.groupBy("YEAR")
      .agg(
          F.count("*").alias("flights"),
          F.avg("DEP_DEL15").alias("delay_rate")
      )
      .orderBy("YEAR")
)

# display(year_delay)

year_pd = year_delay.toPandas()

plt.figure(figsize=(10,5))
plt.bar(year_pd["YEAR"].astype(str), year_pd["delay_rate"])
plt.title("Delay Rate by Year")
plt.ylabel("Delay Rate")
plt.xlabel("Year")
plt.show()

## Delay by Month

In [0]:

month_delay = (
    df_flights_usable.groupBy("MONTH")
      .agg(
          F.count("*").alias("flights"),
          F.avg("DEP_DEL15").alias("delay_rate")
      )
      .orderBy("MONTH")
)

# display(month_delay)

month_pd = month_delay.toPandas()

plt.figure(figsize=(10,5))
plt.bar(month_pd["MONTH"].astype(str), month_pd["delay_rate"])
plt.title("Delay Rate by Month")
plt.ylabel("Delay Rate")
plt.xlabel("Month")
plt.show()

In [0]:
df_flights_usable.printSchema()

## Delay by Date

In [0]:
date_delay = (
    df_flights_usable.groupBy("FL_DATE")
      .agg(
          F.avg("DEP_DEL15").alias("delay_rate"),
          F.count("*").alias("n_flights")
      )
      .orderBy("FL_DATE")
)


#display(dow_delay)

date_pd = date_delay.toPandas()
date_pd["FL_DATE"] = pd.to_datetime(date_pd["FL_DATE"])

plt.figure(figsize=(12,5))

plt.plot(date_pd["FL_DATE"], date_pd["delay_rate"], marker="o")

plt.title("Delay Rate by Date")
plt.ylabel("Delay Rate")
plt.xlabel("Date")

# Rotate x-axis labels
plt.xticks(rotation=60, ha='right')

plt.tight_layout()  # prevents clipping
plt.show()

In [0]:
plt.figure(figsize=(12,5))

plt.scatter(
    date_pd["n_flights"],
    date_pd["delay_rate"],
    #c=date_pd["n_flights"],
    #cmap="viridis",
    alpha=0.7
)

plt.title("Scatter Plot of Delay Rate by Flight Volume")
plt.ylabel("Delay Rate")
plt.xlabel("Volume of Flights")

#plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()



In [0]:
print("Delay rate when the flight volume is less than the average:")
print(date_pd[date_pd["n_flights"]<date_pd["n_flights"].mean()]["delay_rate"].mean())
print(date_pd[date_pd["n_flights"]<date_pd["n_flights"].mean()]["delay_rate"].var())

print("Delay rate when the flight volume is greater or equal to the average:")
print(date_pd[date_pd["n_flights"]>=date_pd["n_flights"].mean()]["delay_rate"].mean())
print(date_pd[date_pd["n_flights"]>=date_pd["n_flights"].mean()]["delay_rate"].var())

### Previous-Day Delay Propagation

In [0]:
df_with_hour = df_flights_usable.withColumn("hour", (F.col("CRS_DEP_TIME") / 100).cast("int"))

# Calculate the aggregate delay rate for every single day
daily_performance = df_flights_usable.groupBy("FL_DATE").agg(
    F.avg("DEP_DEL15").alias("daily_avg_delay")
)

# Create a Window to 'lag' the data (Get Yesterday's performance for every Today)
# We sort by date to ensure we are looking at the previous chronological day
windowSpec = Window.orderBy("FL_DATE")

daily_performance_lagged = daily_performance.withColumn(
    "yesterday_avg_delay", 
    F.lag("daily_avg_delay", 1).over(windowSpec)
).dropna() # Remove the first day of the dataset as it has no 'yesterday'

# Categorize 'Today' based on how 'Yesterday' performed (using the median)
median_yesterday = daily_performance_lagged.approxQuantile("yesterday_avg_delay", [0.5], 0.01)[0]

day_categories = daily_performance_lagged.withColumn(
    "yesterday_category", 
    F.when(F.col("yesterday_avg_delay") > median_yesterday, "High-Delay Yesterday")
    .otherwise("Low-Delay Yesterday")
).select("FL_DATE", "yesterday_category")

# Join categories back to hourly flight data to see today's trajectory
inter_day_data = (
    df_with_hour.join(day_categories, on="FL_DATE")
    .groupBy("yesterday_category", "hour")
    .agg(F.avg("DEP_DEL15").alias("avg_delay_rate"))
    .orderBy("hour")
).toPandas()

# Visualization
plt.figure(figsize=(12, 6))
pivot_df = inter_day_data.pivot(index='hour', columns='yesterday_category', values='avg_delay_rate')

for category in pivot_df.columns:
    plt.plot(pivot_df.index, pivot_df[category], marker='s', label=category, linewidth=2.5)

plt.title("Inter-Day Delay Propagation", fontsize=14)
plt.xlabel("Hour of Scheduled Departure (Today)", fontsize=12)
plt.ylabel("Average Delay Rate", fontsize=12)
plt.legend(title="Previous Day Performance")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()

### X-Previous-Days Delay Propagation

## Delay by Day of Week

In [0]:
dow_stats = (
    df_flights_usable.groupBy("DAY_OF_WEEK")
      .agg(
          F.avg("DEP_DEL15").alias("delay_rate"),
          F.count("*").alias("n_flights")
      )
      .orderBy("DAY_OF_WEEK")
)

#display(dow_stats)

dow_pd = dow_stats.toPandas()

# Plot
fig, ax1 = plt.subplots(figsize=(10,5))

# Bar plot (delay rate)
ax1.bar(dow_pd["DAY_OF_WEEK"], dow_pd["delay_rate"], alpha=0.6)
ax1.set_ylabel("Delay Rate")
ax1.set_xlabel("Day of Week")
ax1.set_title("Delay Rate and Flight Volume by Day of Week")

# Second axis (flight volume)
ax2 = ax1.twinx()
ax2.plot(dow_pd["DAY_OF_WEEK"], dow_pd["n_flights"], marker="o")
ax2.set_ylabel("Number of Flights")

plt.show()

## Delay by Departure Hour

In [0]:
df_hour = df_flights_usable.withColumn(
    "DEP_HOUR",
    F.floor(F.col("CRS_DEP_TIME") / 100)
)

hour_delay = (
    df_hour.groupBy("DEP_HOUR")
           .agg(
               F.avg("DEP_DEL15").alias("delay_rate"),
               F.count("*").alias("n_flights")
            )
           
           .orderBy("DEP_HOUR")
)

#display(hour_delay)

hour_pd = hour_delay.toPandas()

fig, ax1 = plt.subplots(figsize=(10,5))

# Left axis → delay rate
ax1.plot(hour_pd["DEP_HOUR"], hour_pd["delay_rate"], marker="o")
ax1.set_xlabel("Hour")
ax1.set_ylabel("Delay Rate")
ax1.set_title("Delay Rate and Flight Volume by Departure Hour")

# Right axis → flight volume
ax2 = ax1.twinx()
ax2.plot(hour_pd["DEP_HOUR"], hour_pd["n_flights"], linestyle="--")
ax2.set_ylabel("Number of Flights")

plt.show()

### Intra-day Delay Propagation

In [0]:
morning_delay_start_hour = 0
morning_delay_end_hour = 9

df_hourly = df_flights_usable.withColumn("hour", (F.col("CRS_DEP_TIME") / 100).cast("int"))

# Calculate the 'Morning Baseline' for every day (Hours 5 to 9)
morning_stats = (
    df_hourly.filter((F.col("hour") >= morning_delay_start_hour) & (F.col("hour") <= morning_delay_end_hour))
    .groupBy("FL_DATE")
    .agg(F.avg("DEP_DEL15").alias("morning_delay_rate"))
)

# Categorize days based on the median morning delay
median_morning = morning_stats.approxQuantile("morning_delay_rate", [0.5], 0.01)[0]

day_categories = morning_stats.withColumn(
    "day_type", 
    F.when(F.col("morning_delay_rate") > median_morning, "High-Delay Morning")
    .otherwise("Low-Delay Morning")
)

# Join categories back to the full dataset and aggregate by hour
propagation_data = (
    df_hourly.join(day_categories, on="FL_DATE")
    .groupBy("day_type", "hour")
    .agg(F.avg("DEP_DEL15").alias("avg_delay_rate"))
    .orderBy("hour")
).toPandas() 

# Visualization
plt.figure(figsize=(12, 6))

# Pivot the data for easy line plotting
pivot_df = propagation_data.pivot(index='hour', columns='day_type', values='avg_delay_rate')

for category in pivot_df.columns:
    plt.plot(pivot_df.index, pivot_df[category], marker='o', label=category, linewidth=2.5)

# Highlight the morning window we used for classification
plt.axvspan(morning_delay_start_hour, morning_delay_end_hour, color='gray', alpha=0.15, label='Morning Observation Window')

plt.title("Intra-Day Delay Propagation", fontsize=14)
plt.xlabel("Hour of Scheduled Departure", fontsize=12)
plt.ylabel("Average Delay Rate ($0.0$ to $1.0$)", fontsize=12)
plt.legend(title="Morning Performance")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()

### 24-hour Delay Propagation

In [0]:
24*60*60

In [0]:
past_hours = 24
range_start = -(past_hours*60*60)

df_ts = df_flights_usable.withColumn(
    "timestamp", 
    F.to_timestamp(
        F.concat(F.col("FL_DATE"), F.lit(" "), 
                 F.lpad(F.col("CRS_DEP_TIME"), 4, '0')), 
        "yyyy-MM-dd HHmm"
    )
).withColumn("hour_of_day", (F.col("CRS_DEP_TIME") / 100).cast("int"))

# Aggregate to get the average delay for every unique hour in the dataset
hourly_stats = df_ts.groupBy("timestamp").agg(
    F.avg("DEP_DEL15").alias("current_hour_delay")
)

# Define the Window: Look back exactly 24 hours (86,400 seconds) but exclude the current hour (-1 second)
window_24h = Window.orderBy(F.col("timestamp").cast("long")).rangeBetween(range_start, -1)

# Calculate the Trailing 24h Average
rolling_df = hourly_stats.withColumn(
    "trailing_24h_delay", 
    F.avg("current_hour_delay").over(window_24h)
).dropna()

# Categorize based on the median of the trailing 24h delay
median_trailing = rolling_df.approxQuantile("trailing_24h_delay", [0.5], 0.01)[0]

rolling_categories = rolling_df.withColumn(
    "lookback_category", 
    F.when(F.col("trailing_24h_delay") > median_trailing, "High-Stress (Trailing 24h)")
    .otherwise("Low-Stress (Trailing 24h)")
)

# Join back to the full flight data to see the "Current" impact by hour of day
final_plot_data = (
    df_ts.join(rolling_categories.select("timestamp", "lookback_category"), on="timestamp")
    .groupBy("lookback_category", "hour_of_day")
    .agg(F.avg("DEP_DEL15").alias("avg_delay_rate"))
    .orderBy("hour_of_day")
).toPandas()

# Visualization
plt.figure(figsize=(12, 6))
pivot_df = final_plot_data.pivot(index='hour_of_day', columns='lookback_category', values='avg_delay_rate')

for category in pivot_df.columns:
    plt.plot(pivot_df.index, pivot_df[category], marker='d', label=category, linewidth=2.5)

plt.title("24-Hour Delay Propagation", fontsize=14)
plt.xlabel("Current Hour of Departure", fontsize=12)
plt.ylabel("Average Delay Rate", fontsize=12)
plt.legend(title="Last 24h Performance")
plt.grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()

## Delay by Arrival Hour

In [0]:
df_arr_hour = df_flights_usable.withColumn(
    "ARR_HOUR",
    F.floor(F.col("CRS_ARR_TIME") / 100)
)

arr_hour_delay = (
    df_arr_hour.groupBy("ARR_HOUR")
           .agg(F.avg("DEP_DEL15").alias("delay_rate"),
                F.count("*").alias("n_flights")
                )
           .orderBy("ARR_HOUR")
)
arr_hour_delay = arr_hour_delay.filter("ARR_HOUR < 24")

#display(arr_hour_delay)

arr_hour_pd = arr_hour_delay.toPandas()

fig, ax1 = plt.subplots(figsize=(10,5))

# Left axis → delay rate
ax1.plot(arr_hour_pd["ARR_HOUR"], arr_hour_pd["delay_rate"], marker="o")
ax1.set_xlabel("Hour")
ax1.set_ylabel("Delay Rate")
ax1.set_title("Delay Rate and Flight Volume by Arrival Hour")

# Right axis → flight volume
ax2 = ax1.twinx()
ax2.plot(arr_hour_pd["ARR_HOUR"], arr_hour_pd["n_flights"], linestyle="--")
ax2.set_ylabel("Number of Flights")

plt.show()

## Delay by Day of Month

In [0]:
dom_delay = (
    df_flights_usable.groupBy("DAY_OF_MONTH")
      .agg(F.avg("DEP_DEL15").alias("delay_rate"))
      .orderBy("DAY_OF_MONTH")
)

#display(dow_delay)

dom_pd = dom_delay.toPandas()

plt.figure(figsize=(10,5))
plt.bar(dom_pd["DAY_OF_MONTH"], dom_pd["delay_rate"])
plt.title("Delay Rate by Day of Month")
plt.ylabel("Delay Rate")
plt.xlabel("Day of Month")
plt.show()

## Delay by Seasonality

[To-Do] Find all the US holiday dates from 2015 on

In [0]:
national_seasonality = {"date": "1",
                        }

## Delay by Airline

In [0]:
carrier_delay = (
    df_flights_usable.groupBy("OP_UNIQUE_CARRIER")
      .agg(
          F.count("*").alias("flights"),
          F.avg("DEP_DEL15").alias("delay_rate")
      )
      .orderBy(F.desc("delay_rate"))
)

# display(carrier_delay)

carrier_pd = carrier_delay.toPandas()

plt.figure(figsize=(10,5))
plt.bar(carrier_pd["OP_UNIQUE_CARRIER"], carrier_pd["delay_rate"])
plt.title("Delay Rate by Airline")
plt.ylabel("Delay Rate")
plt.xlabel("Carrier")
plt.show()

In [0]:
print(carrier_pd.head(1).iloc[0]["OP_UNIQUE_CARRIER"], carrier_pd[carrier_pd["OP_UNIQUE_CARRIER"]==carrier_pd.head(1).iloc[0]["OP_UNIQUE_CARRIER"]]["delay_rate"])

print(carrier_pd.tail(1).iloc[0]["OP_UNIQUE_CARRIER"], carrier_pd[carrier_pd["OP_UNIQUE_CARRIER"]==carrier_pd.tail(1).iloc[0]["OP_UNIQUE_CARRIER"]]["delay_rate"])
carrier_pd["delay_rate"].mean()

## Airport Delay Hotspots


In [0]:

origin_delay = (
    df_flights_usable.groupBy("ORIGIN")
      .agg(
          F.count("*").alias("flights"),
          F.avg("DEP_DEL15").alias("delay_rate")
      )
      .filter("flights > 5000")
      .orderBy(F.desc("delay_rate"))
)

#display(origin_delay.limit(20))

origin_pd = origin_delay.toPandas()

fig, ax1 = plt.subplots(figsize=(12,5))

# Left axis → delay rate (bars)
ax1.bar(origin_pd["ORIGIN"], origin_pd["delay_rate"], alpha=0.6)
ax1.set_ylabel("Delay Rate")
ax1.set_xlabel("Origin")
ax1.set_title("Delay Rate and Flight Volume by Origin")

# Rotate x-axis labels
plt.setp(ax1.get_xticklabels(), rotation=45, ha='right')

# Right axis → number of flights (line)
ax2 = ax1.twinx()
ax2.plot(origin_pd["ORIGIN"], origin_pd["flights"], linestyle="--", marker="o")
ax2.set_ylabel("Number of Flights")

plt.tight_layout()
plt.show()

In [0]:
print(origin_pd.head(1).iloc[0]["ORIGIN"], origin_pd[origin_pd["ORIGIN"]==origin_pd.head(1).iloc[0]["ORIGIN"]]["delay_rate"])

print(origin_pd.tail(1).iloc[0]["ORIGIN"], origin_pd[origin_pd["ORIGIN"]==origin_pd.tail(1).iloc[0]["ORIGIN"]]["delay_rate"])
origin_pd["delay_rate"].mean()

## Outlier Detection

In [0]:
q = df_flights_usable.approxQuantile("DEP_DELAY",[0.25,0.75],0.01)
iqr = q[1] - q[0]

lower = q[0] - 1.5*iqr
upper = q[1] + 1.5*iqr

outliers = df_flights_usable.filter((F.col("DEP_DELAY") < lower) | (F.col("DEP_DELAY") > upper)).count()

print("Outliers:", outliers)

## Correlation matrix of unnormalized values

In [0]:
numeric_types = (IntegerType, DoubleType, FloatType, LongType, ShortType, DecimalType)

numeric_cols = [
    field.name
    for field in df_flights_usable.schema.fields
    if isinstance(field.dataType, numeric_types)
]

not_include = set(["OP_CARRIER_AIRLINE_ID", "OP_CARRIER_FL_NUM", "DEP_DELAY"])
numeric_cols = [c for c in numeric_cols if c not in not_include]
print(numeric_cols)

In [0]:
df_sample = df_flights_usable.select(numeric_cols)
#print(df_sample.dtypes)
df_sample = df_sample.toPandas()
corr_matrix = df_sample.corr()

In [0]:
plt.figure(figsize=(12,10))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Correlation Heatmap - Numeric Features")
plt.show()

## Correlation matrix with standarized columns

In [0]:
df_sample = df_sample.dropna(axis=1, how="all")
df_sample = df_sample.loc[:, df_sample.nunique(dropna=True) > 1]
df_standardized = (df_sample - df_sample.mean()) / df_sample.std(ddof=0)
df_standardized = df_standardized.dropna(axis=1, how="all")

corr_matrix = df_standardized.corr()

plt.figure(figsize=(12, 10))
sns.heatmap(
    corr_matrix,
    cmap="coolwarm",
    center=0,
    vmin=-1,
    vmax=1,
    linewidths=0.3
)
plt.title("Correlation Heatmap of Standardized Numeric Features")

## Origin-destination graph

In [0]:
df_airports = spark.read.parquet(
    "dbfs:/mnt/mids-w261/datasets_final_project_2022/stations_data/stations_with_neighbors.parquet/"
)
df_airports = df_airports.cache()

The airport reference dataset used ICAO codes, which include a regional prefix (e.g., “K” for U.S. mainland, “P” for Alaska). Since the flight dataset uses IATA codes, a transformation was applied to convert ICAO to IATA by removing the leading “K” for U.S. airports. To avoid incorrect mappings (e.g., “PATL” in Alaska vs “KATL” in Georgia), the dataset was first filtered to include only ICAO codes starting with “K”.

In [0]:
df_airports_iata = (
    df_airports
    .filter(F.col("neighbor_call").startswith("K"))  # only continental US
    .withColumn(
        "AIRPORT_CODE_3",
        F.expr("substring(neighbor_call, 2, 3)")
    )
)

df_airports_iata = df_airports_iata.select(
        F.col("AIRPORT_CODE_3").alias("AIRPORT"),
        F.col("neighbor_lat").alias("LAT"),
        F.col("neighbor_lon").alias("LON")
    ).filter(
        F.col("AIRPORT").isNotNull() &
        F.col("LAT").isNotNull() &
        F.col("LON").isNotNull()
    ).dropDuplicates(["AIRPORT"])

In [0]:
routes = (
    df_flights_usable
    .groupBy("ORIGIN", "DEST")
    .agg(F.count("*").alias("n_flights"))
    .filter(F.col("ORIGIN").isNotNull() & F.col("DEST").isNotNull())
)

origin_lookup = (
    df_airports_iata
    .select(
        F.col("AIRPORT").alias("ORIGIN"),
        F.col("LAT").alias("ORIGIN_LAT"),
        F.col("LON").alias("ORIGIN_LON")
    )
)

dest_lookup = (
    df_airports_iata
    .select(
        F.col("AIRPORT").alias("DEST"),
        F.col("LAT").alias("DEST_LAT"),
        F.col("LON").alias("DEST_LON")
    )
)

routes_geo = (
    routes
    .join(origin_lookup, on="ORIGIN", how="inner")
    .join(dest_lookup, on="DEST", how="inner")
)

In [0]:
routes_top = (
    routes_geo
    .orderBy(F.desc("n_flights"))
    .limit(400)
)

routes_pd = routes_top.toPandas()
#routes_pd.head()

In [0]:
routes_pd.head()

In [0]:
for c in ["ORIGIN_LAT", "ORIGIN_LON", "DEST_LAT", "DEST_LON", "n_flights"]:
    routes_pd[c] = pd.to_numeric(routes_pd[c], errors="coerce")

routes_pd = routes_pd.dropna(subset=["ORIGIN_LAT", "ORIGIN_LON", "DEST_LAT", "DEST_LON"]).copy()
routes_pd["arc_width"] = routes_pd["n_flights"].clip(lower=1)
routes_pd["tooltip_text"] = routes_pd.apply(
    lambda row: f"{row['ORIGIN']} → {row['DEST']}: {int(row['n_flights'])} flights",
    axis=1
)

airport_points = pd.concat([
    routes_pd[["ORIGIN", "ORIGIN_LAT", "ORIGIN_LON"]].rename(
        columns={"ORIGIN": "AIRPORT", "ORIGIN_LAT": "LAT", "ORIGIN_LON": "LON"}
    ),
    routes_pd[["DEST", "DEST_LAT", "DEST_LON"]].rename(
        columns={"DEST": "AIRPORT", "DEST_LAT": "LAT", "DEST_LON": "LON"}
    )
]).drop_duplicates()

origin_points = (
    routes_pd.groupby(["ORIGIN", "ORIGIN_LAT", "ORIGIN_LON"], as_index=False)["n_flights"]
    .sum()
    .rename(columns={
        "ORIGIN": "AIRPORT",
        "ORIGIN_LAT": "LAT",
        "ORIGIN_LON": "LON",
        "n_flights": "traffic"
    })
)

dest_points = (
    routes_pd.groupby(["DEST", "DEST_LAT", "DEST_LON"], as_index=False)["n_flights"]
    .sum()
    .rename(columns={
        "DEST": "AIRPORT",
        "DEST_LAT": "LAT",
        "DEST_LON": "LON",
        "n_flights": "traffic"
    })
)

airport_activity = pd.concat([origin_points, dest_points], ignore_index=True)
airport_activity = (
    airport_activity
    .groupby(["AIRPORT", "LAT", "LON"], as_index=False)["traffic"]
    .sum()
)

In [0]:
heat_layer = pdk.Layer(
    "HeatmapLayer",
    data=airport_activity,
    get_position="[LON, LAT]",
    get_weight="traffic",
    aggregation="SUM",
    opacity=0.7,
    radiusPixels=50
)
arc_layer = pdk.Layer(
    "ArcLayer",
    data=routes_pd,
    get_source_position="[ORIGIN_LON, ORIGIN_LAT]",
    get_target_position="[DEST_LON, DEST_LAT]",
    get_width="arc_width",
    width_scale=0.002,
    get_source_color=[0, 255, 0, 120],      # green
    get_target_color=[255, 140, 0, 150],    # orange-red
    pickable=True,
    auto_highlight=True,
    blend=True
)

view_state = pdk.ViewState(
    latitude=39.5,
    longitude=-98.35,
    zoom=3.4,
    pitch=45,
    bearing=0
)

deck = pdk.Deck(
    layers=[heat_layer, arc_layer],
    initial_view_state=view_state,
    map_provider="carto",
    map_style="dark",
    tooltip={"text": "{tooltip_text}"}
)

display(deck.show())

In [0]:
origin_dest_delay = (
    df_flights_usable.groupBy(["ORIGIN", "DEST"]).agg(
          F.count("*").alias("flights"),
          F.avg("DEP_DEL15").alias("delay_rate")
      )
      .filter(F.col("flights") > 10) 
)

origin_dest_pd = origin_dest_delay.toPandas()

top_routes = origin_dest_pd.nlargest(5, 'flights')

plt.figure(figsize=(10,5))
plt.scatter(
    origin_dest_pd["flights"], 
    origin_dest_pd["delay_rate"], 
    alpha=0.4, 
    c='royalblue', 
    edgecolors='white'
)

# 3. Annotate the top routes
for i, row in top_routes.iterrows():
    label = f"{row['ORIGIN']}->{row['DEST']}"
    plt.annotate(
        label, 
        (row['flights'], row['delay_rate']),
        textcoords="offset points", 
        xytext=(2,2), # Position the text slightly above/right of the point
        ha='left',
        fontsize=8,
        #fontweight='bold',
        #bbox=dict(boxstyle="round,pad=0.3", fc="white", alpha=0.8)
    )

plt.title("Flight Volume vs. Average Delay Rate (segmented by route)")
plt.xlabel("Volume of Flights")
plt.ylabel("Average Delay Rate")
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()